# Week 25, Day 1: Project: expense tracker

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w025d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


> **Key idea.** Plan the data first: one dictionary of expenses, budgets and the next id, ready for JSON. Money is whole pence, dates are ISO text, and ids come from a stored counter so they're never reused. Only the interface prints; the rest takes values and returns values.

### Exercise: Step 1: read an amount

Write `parse_amount(text)` returning the amount in whole pence. Strip spaces first. Accept an optional `£` at the start, digits with optional commas between groups of three (`1,250` but not `12,50`), and an optional `.` with 1 or 2 digits. Raise `ValueError` with:

- `missing amount` for empty text (or just spaces, or just `£`);
- `not an amount: '12.345'` (the stripped text, as `{text!r}` shows it) if it doesn't fit the pattern;
- `amount must be more than £0` for zero.

In [ ]:
import re

AMOUNT = re.compile(r"___")

def parse_amount(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def message(text):\n    try:\n        parse_amount(text)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"parse_amount({text!r}) should raise ValueError\")\ndef test_good():\n    \"good amounts\"\n    cases = {\"12\": 1200, \"12.5\": 1250, \"12.50\": 1250, \"\u00a312.50\": 1250, \" \u00a3 3.05 \": 305, \"1,250.00\": 125000,\n             \"0.01\": 1, \"3,200\": 320000, \"1,000,000\": 100000000}\n    for text, want in cases.items():\n        got = parse_amount(text)\n        assert got == want and type(got) is int, f\"parse_amount({text!r}) should be {want}, not {got!r}\"\ndef test_missing():\n    \"missing amounts\"\n    for t in (\"\", \"   \", \"\u00a3\", \" \u00a3 \"):\n        assert message(t) == \"missing amount\", f\"parse_amount({t!r})\"\ndef test_bad():\n    \"not amounts\"\n    for t, shown in ((\"12.345\", \"'12.345'\"), (\"abc\", \"'abc'\"), (\"-3\", \"'-3'\"), (\"12,50\", \"'12,50'\"),\n                     (\" 7. \", \"'7.'\"), (\"1.2.3\", \"'1.2.3'\"), (\"\u00a3 1 0\", \"'\u00a3 1 0'\"), (\"$5\", \"'$5'\")):\n        assert message(t) == f\"not an amount: {shown}\", f\"parse_amount({t!r}) gave {message(t)!r}\"\ndef test_zero():\n    \"zero\"\n    for t in (\"0\", \"0.00\", \"\u00a30.0\"):\n        assert message(t) == \"amount must be more than \u00a30\"")

### Exercise: Step 2: show an amount

Write `format_pence(pence)` that shows a whole number of pence as pounds: `1250` as `£12.50`, `5` as `£0.05`, `320000` as `£3,200.00`, `0` as `£0.00`. Negative amounts (a budget overspent, say) get a minus sign before the `£`: `-350` is `-£3.50`.

In [ ]:
import re

AMOUNT = re.compile(r"£?\s*(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d{1,2}))?")

def parse_amount(text):
    text = text.strip()
    if not text.lstrip("£").strip():
        raise ValueError("missing amount")
    m = AMOUNT.fullmatch(text)
    if m is None:
        raise ValueError(f"not an amount: {text!r}")
    pounds = int(m.group(1).replace(",", ""))
    pence = int((m.group(2) or "").ljust(2, "0"))
    total = pounds * 100 + pence
    if total == 0:
        raise ValueError("amount must be more than £0")
    return total

def format_pence(pence):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"amounts\"\n    cases = {1250: \"\u00a312.50\", 5: \"\u00a30.05\", 320000: \"\u00a33,200.00\", 0: \"\u00a30.00\", 100: \"\u00a31.00\",\n             -350: \"-\u00a33.50\", 123456789: \"\u00a31,234,567.89\", -5: \"-\u00a30.05\"}\n    for p, want in cases.items():\n        assert format_pence(p) == want, f\"format_pence({p}) should be {want!r}, not {format_pence(p)!r}\"\ndef test_round_trip():\n    \"parse_amount(format_pence(p)) == p for many amounts\"\n    for p in list(range(1, 300)) + [99999, 100000, 1234567]:\n        assert parse_amount(format_pence(p)) == p, p")

### Exercise: Step 3: read a date

Write `parse_date(text, today)` returning the date as ISO text (`"2026-11-02"`), where `today` is a `date`. Strip and lower-case the text, then:

- empty text means today;
- `today` and `yesterday` mean what they say;
- otherwise try `%Y-%m-%d`, then `%d/%m/%Y`; if neither fits, raise `ValueError("not a date: '31/02/2026'")` (the stripped text, as `{text!r}` shows it, before lower-casing);
- a date after `today` raises `ValueError("date is in the future: 2026-11-05")`.

In [ ]:
from datetime import date, datetime, timedelta

FORMATS = ["%Y-%m-%d", "%d/%m/%Y"]

def parse_date(text, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _date\nT = _date(2026, 11, 2)\ndef message(text):\n    try:\n        parse_date(text, T)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"parse_date({text!r}, today) should raise ValueError\")\ndef test_words():\n    \"empty, today and yesterday\"\n    assert parse_date(\"\", T) == \"2026-11-02\" and parse_date(\"  \", T) == \"2026-11-02\"\n    assert parse_date(\"today\", T) == \"2026-11-02\" and parse_date(\" Yesterday \", T) == \"2026-11-01\"\n    assert parse_date(\"yesterday\", _date(2027, 1, 1)) == \"2026-12-31\"\ndef test_formats():\n    \"both formats\"\n    assert parse_date(\"2026-10-31\", T) == \"2026-10-31\"\n    assert parse_date(\"31/10/2026\", T) == \"2026-10-31\"\n    assert parse_date(\" 1/9/2026 \", T) == \"2026-09-01\"\n    assert parse_date(\"2026-11-02\", T) == \"2026-11-02\"\ndef test_bad():\n    \"not dates\"\n    assert message(\"31/02/2026\") == \"not a date: '31/02/2026'\"\n    assert message(\"Tomorrow\") == \"not a date: 'Tomorrow'\"\n    assert message(\"2026/11/01\") == \"not a date: '2026/11/01'\"\n    assert message(\"11-02-2026\") == \"not a date: '11-02-2026'\"\ndef test_future():\n    \"future dates\"\n    assert message(\"2026-11-05\") == \"date is in the future: 2026-11-05\"\n    assert message(\"03/11/2026\") == \"date is in the future: 2026-11-03\"")

### Exercise: Step 4: clean a category

Write `clean_category(text)`: strip, lower-case, and collapse runs of spaces to one. The result must be one or more words of letters `a` to `z` separated by single spaces, and at most 20 characters. Raise `ValueError("missing category")` for empty text, and `ValueError("bad category: 'Food & Drink'")` (the stripped text as typed) otherwise.

In [ ]:
import re

def clean_category(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def message(text):\n    try:\n        clean_category(text)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"clean_category({text!r}) should raise ValueError\")\ndef test_good():\n    \"good categories\"\n    assert clean_category(\"food\") == \"food\" and clean_category(\" Eating   Out \") == \"eating out\"\n    assert clean_category(\"RENT\") == \"rent\" and clean_category(\"a\" * 20) == \"a\" * 20\n    assert clean_category(\"days\\tout\") == \"days out\"\ndef test_bad():\n    \"bad categories\"\n    assert message(\"\") == \"missing category\" and message(\"   \") == \"missing category\"\n    assert message(\" Food & Drink \") == \"bad category: 'Food & Drink'\"\n    assert message(\"bills2\") == \"bad category: 'bills2'\"\n    assert message(\"a\" * 21) == f\"bad category: {'a' * 21!r}\"\n    assert message(\"caf\u00e9\") == \"bad category: 'caf\u00e9'\"")

### Exercise: Step 5: add an expense

Write `new_data()` returning an empty data dictionary (`{"expenses": [], "budgets": {}, "next_id": 1}`), and `add_expense(data, amount_text, category_text, note, date_text, today)`. It:

1. checks the amount, the category and the date with the earlier functions, in that order, and the note: stripped, at most 60 characters, otherwise the problem `note is longer than 60 characters`;
2. if there are any problems, raises **one** `ValueError` whose message is all of them joined by `"; "`, and changes nothing;
3. otherwise appends `{"id", "date", "amount", "category", "note"}` (in that order) with the id from `next_id`, adds 1 to `next_id`, and returns the new expense.

In [ ]:
import re
from datetime import date, datetime, timedelta

AMOUNT = re.compile(r"£?\s*(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d{1,2}))?")
FORMATS = ["%Y-%m-%d", "%d/%m/%Y"]

def parse_amount(text):
    text = text.strip()
    if not text.lstrip("£").strip():
        raise ValueError("missing amount")
    m = AMOUNT.fullmatch(text)
    if m is None:
        raise ValueError(f"not an amount: {text!r}")
    pounds = int(m.group(1).replace(",", ""))
    pence = int((m.group(2) or "").ljust(2, "0"))
    total = pounds * 100 + pence
    if total == 0:
        raise ValueError("amount must be more than £0")
    return total

def format_pence(pence):
    sign = "-" if pence < 0 else ""
    pounds, rest = divmod(abs(pence), 100)
    return f"{sign}£{pounds:,}.{rest:02d}"

def parse_date(text, today):
    shown = text.strip()
    word = shown.lower()
    if word in ("", "today"):
        when = today
    elif word == "yesterday":
        when = today - timedelta(days=1)
    else:
        when = None
        for fmt in FORMATS:
            try:
                when = datetime.strptime(word, fmt).date()
                break
            except ValueError:
                pass
        if when is None:
            raise ValueError(f"not a date: {shown!r}")
    if when > today:
        raise ValueError(f"date is in the future: {when.isoformat()}")
    return when.isoformat()

def clean_category(text):
    shown = text.strip()
    if not shown:
        raise ValueError("missing category")
    cleaned = " ".join(shown.lower().split())
    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):
        raise ValueError(f"bad category: {shown!r}")
    return cleaned

def new_data():
    ___

def add_expense(data, amount_text, category_text, note, date_text, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _date\nT = _date(2026, 11, 2)\ndef test_new():\n    \"new_data\"\n    assert new_data() == {\"expenses\": [], \"budgets\": {}, \"next_id\": 1}\n    a, b = new_data(), new_data()\n    a[\"expenses\"].append(1)\n    assert b[\"expenses\"] == [], \"each call should give a new dictionary\"\ndef test_add():\n    \"adding\"\n    d = new_data()\n    e = add_expense(d, \"12.50\", \" Food \", \"  lunch with Tola \", \"\", T)\n    assert e == {\"id\": 1, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\", \"note\": \"lunch with Tola\"}\n    assert list(e) == [\"id\", \"date\", \"amount\", \"category\", \"note\"]\n    e2 = add_expense(d, \"\u00a33,200\", \"rent\", \"\", \"01/11/2026\", T)\n    assert e2[\"id\"] == 2 and e2[\"amount\"] == 320000 and d[\"next_id\"] == 3 and d[\"expenses\"] == [e, e2]\ndef test_all_problems():\n    \"every problem, in order, and nothing changed\"\n    d = new_data()\n    add_expense(d, \"1\", \"food\", \"\", \"\", T)\n    before = {\"expenses\": list(d[\"expenses\"]), \"next_id\": d[\"next_id\"]}\n    try:\n        add_expense(d, \"12.345\", \"Food & Drink\", \"x\" * 61, \"tomorrow\", T)\n        assert False, \"should raise ValueError\"\n    except ValueError as e:\n        assert str(e) == (\"not an amount: '12.345'; bad category: 'Food & Drink'; \"\n                          \"not a date: 'tomorrow'; note is longer than 60 characters\"), str(e)\n    assert d[\"expenses\"] == before[\"expenses\"] and d[\"next_id\"] == before[\"next_id\"]\ndef test_one_problem():\n    \"a single problem\"\n    d = new_data()\n    try:\n        add_expense(d, \"5\", \"food\", \"\", \"2026-12-25\", T)\n        assert False, \"should raise ValueError\"\n    except ValueError as e:\n        assert str(e) == \"date is in the future: 2026-12-25\"\n    assert d == new_data()\n    assert add_expense(d, \"5\", \"food\", \"x\" * 60, \"\", T)[\"note\"] == \"x\" * 60")

> **All or nothing.** `add_expense` changes `data` only after every check has passed. If it appended first and checked afterwards, a failed check would leave half an entry behind, or an entry with no amount. "Check everything, then change everything" is a habit worth keeping for any function that changes data, and it's what databases call a **transaction**.

### Exercise: Step 6: remove an expense

Write `remove_expense(data, expense_id)` that removes the expense with that id and returns it, or raises `KeyError` with the message `no expense #7` if there isn't one. `next_id` must **not** change.

To check it in the editor, the starter includes a compact `new_data` and two expenses to work with.

In [ ]:
def new_data():
    return {"expenses": [], "budgets": {}, "next_id": 1}

def remove_expense(data, expense_id):
    ___

data = new_data()
data["expenses"] = [
    {"id": 1, "date": "2026-11-01", "amount": 320000, "category": "rent", "note": ""},
    {"id": 2, "date": "2026-11-02", "amount": 1250, "category": "food", "note": "lunch"},
]
data["next_id"] = 3
print(remove_expense(data, 2))
print(data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _data():\n    return {\"expenses\": [\n        {\"id\": 1, \"date\": \"2026-11-01\", \"amount\": 320000, \"category\": \"rent\", \"note\": \"\"},\n        {\"id\": 4, \"date\": \"2026-11-02\", \"amount\": 1250, \"category\": \"food\", \"note\": \"lunch\"},\n        {\"id\": 5, \"date\": \"2026-11-02\", \"amount\": 300, \"category\": \"food\", \"note\": \"\"},\n    ], \"budgets\": {}, \"next_id\": 6}\ndef test_remove():\n    \"removes and returns it\"\n    d = _data()\n    e = remove_expense(d, 4)\n    assert e[\"id\"] == 4 and e[\"note\"] == \"lunch\"\n    assert [x[\"id\"] for x in d[\"expenses\"]] == [1, 5] and d[\"next_id\"] == 6\ndef test_remove_newest():\n    \"removing the newest doesn't change next_id\"\n    d = _data()\n    remove_expense(d, 5)\n    assert d[\"next_id\"] == 6\ndef test_missing():\n    \"an unknown id raises KeyError, and nothing changes\"\n    d = _data()\n    for bad in (2, 0, 6):\n        try:\n            remove_expense(d, bad)\n            assert False, f\"remove_expense(data, {bad}) should raise KeyError\"\n        except KeyError as e:\n            assert e.args[0] == f\"no expense #{bad}\"\n    assert d == _data()")

In [ ]:
import re
from datetime import date, datetime, timedelta

AMOUNT = re.compile(r"£?\s*(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d{1,2}))?")
FORMATS = ["%Y-%m-%d", "%d/%m/%Y"]

def parse_amount(text):
    text = text.strip()
    if not text.lstrip("£").strip():
        raise ValueError("missing amount")
    m = AMOUNT.fullmatch(text)
    if m is None:
        raise ValueError(f"not an amount: {text!r}")
    total = int(m.group(1).replace(",", "")) * 100 + int((m.group(2) or "").ljust(2, "0"))
    if total == 0:
        raise ValueError("amount must be more than £0")
    return total

def format_pence(pence):
    sign = "-" if pence < 0 else ""
    pounds, rest = divmod(abs(pence), 100)
    return f"{sign}£{pounds:,}.{rest:02d}"

def parse_date(text, today):
    shown = text.strip()
    word = shown.lower()
    if word in ("", "today"):
        when = today
    elif word == "yesterday":
        when = today - timedelta(days=1)
    else:
        when = None
        for fmt in FORMATS:
            try:
                when = datetime.strptime(word, fmt).date()
                break
            except ValueError:
                pass
        if when is None:
            raise ValueError(f"not a date: {shown!r}")
    if when > today:
        raise ValueError(f"date is in the future: {when.isoformat()}")
    return when.isoformat()

def clean_category(text):
    shown = text.strip()
    if not shown:
        raise ValueError("missing category")
    cleaned = " ".join(shown.lower().split())
    if len(cleaned) > 20 or not re.fullmatch(r"[a-z]+( [a-z]+)*", cleaned):
        raise ValueError(f"bad category: {shown!r}")
    return cleaned

def new_data():
    return {"expenses": [], "budgets": {}, "next_id": 1}

def add_expense(data, amount_text, category_text, note, date_text, today):
    problems = []
    amount = category = when = None
    for check in ("amount", "category", "date"):
        try:
            if check == "amount":
                amount = parse_amount(amount_text)
            elif check == "category":
                category = clean_category(category_text)
            else:
                when = parse_date(date_text, today)
        except ValueError as e:
            problems.append(str(e))
    note = note.strip()
    if len(note) > 60:
        problems.append("note is longer than 60 characters")
    if problems:
        raise ValueError("; ".join(problems))
    expense = {"id": data["next_id"], "date": when, "amount": amount, "category": category, "note": note}
    data["expenses"].append(expense)
    data["next_id"] += 1
    return expense

def remove_expense(data, expense_id):
    for i, e in enumerate(data["expenses"]):
        if e["id"] == expense_id:
            return data["expenses"].pop(i)
    raise KeyError(f"no expense #{expense_id}")

def describe(e):
    note = f" ({e['note']})" if e["note"] else ""
    return f"#{e['id']}: {format_pence(e['amount'])} {e['category']}{note} on {e['date']}"

today = date(2026, 11, 2)
data = new_data()
for args in [("3,200", "Rent", "", "01/11/2026"), ("12.50", "food", "lunch with Tola", ""),
             ("4.2", " Travel ", "bus", "yesterday"), ("12.345", "Food & Drink", "", "tomorrow")]:
    try:
        print("Added", describe(add_expense(data, *args, today)))
    except ValueError as e:
        print("Not added:", e)

print("Removed", describe(remove_expense(data, 3)))
print("Added", describe(add_expense(data, "2.80", "food", "coffee", "", today)))
print(data["next_id"], [e["id"] for e in data["expenses"]])

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## The brief

You want to know where your money goes. Spreadsheets work, but they're slow to update from a phone keyboard or a terminal, and easy to mistype. So you'll write a small tool:

```
$ python3 spend.py add 12.50 food "lunch with Tola"
Added #14: £12.50 food (lunch with Tola) on 2026-11-02
$ python3 spend.py add 3,200 rent --date 01/11/2026
Added #15: £3,200.00 rent on 2026-11-01
$ python3 spend.py list --month 2026-11
$ python3 spend.py summary --month 2026-11
$ python3 spend.py budget food 150
$ python3 spend.py import statement.csv
```

What it must do, by the end of the week:

1. **Add** an expense: an amount, a category and an optional note and date (today if not given). Bad input is rejected with a message saying **everything** that's wrong, and nothing is saved.
2. **Remove** an expense by its number.
3. **List** expenses, filtered by month and category, newest first.
4. **Summarise** a month: the total, the total per category, and how each category compares with its budget.
5. **Budgets**: set a monthly budget per category, and warn when spending goes over.
6. **Save** everything to a JSON file, safely, so a crash halfway through saving never loses data.
7. **Import** a bank's CSV export, guessing categories from the description, and skipping rows already imported.
8. **Export** to CSV for a spreadsheet.

Rules for the data:

- An **amount** is more than £0 and has at most 2 decimal places. People may type a `£`, and commas between thousands: `12`, `12.5`, `£12.50` and `1,250.00` are all fine.
- A **date** is written `2026-11-02` or `02/11/2026` (day first), or the words `today` and `yesterday`. It can't be in the future.
- A **category** is one or more words of letters, at most 20 characters, stored in lower case with single spaces: `" Eating   Out "` is stored as `"eating out"`.
- A **note** is optional, at most 60 characters, stripped of spaces at the ends.
- Every expense gets a **number** (an id): 1, 2, 3 and so on. A removed expense's number is **never reused**, so "#14" always means the same expense.

## A plan

Before any code, decide the pieces. Following Week 14's method: understand the problem, plan the data, then split the work into functions, each with one job.

**The data.** Everything lives in one dictionary, which is easy to save as JSON:

```
{
  "expenses": [
    {"id": 1, "date": "2026-11-01", "amount": 320000, "category": "rent", "note": ""},
    {"id": 2, "date": "2026-11-02", "amount": 1250, "category": "food", "note": "lunch with Tola"}
  ],
  "budgets": {"food": 15000},
  "next_id": 3
}
```

Three decisions are hiding in there:

- **Amounts are whole pence**, as `int`s. Floats can't store most decimal fractions exactly (Week 24's `0.1 + 0.2`), and money mustn't drift by a penny after a thousand additions. Integers add exactly. Pounds and pence only appear when reading what a person typed, and when printing.
- **Dates are ISO text**, `"2026-11-02"`. JSON has no date type (Week 23), ISO dates sort correctly as text, and the month is just the first 7 characters.
- **`next_id` is stored**, rather than worked out as "the largest id plus one". If the newest expense is removed, the largest-plus-one would hand its number to the next one; a stored counter never goes back.

**The modules.** By the end of the week the program is split into files by job, as in Week 20:

| Module | Its job | Built on |
|---|---|---|
| `money.py`, `dates.py` | read and show amounts, dates and categories | Day 1 (today) |
| `core.py` | add, remove, filter, total, summarise, budgets | Days 1 and 2 |
| `storage.py` | load and save JSON safely; import and export CSV | Day 3 |
| `spend.py` | the command-line interface: parse arguments, print results | Day 4 |
| `tests/` | pytest tests for all of it | every day |

Only `spend.py` prints or reads arguments. Everything else takes values and returns values (Week 24's "pure core in a thin shell"), so it's easy to test.

In this week's exercises each step's editor starts with the code from the steps before it, already working, so a mistake in one step doesn't follow you into the next.

## Step 1: amounts

Reading an amount is the fiddliest part of the input, so it gets a function of its own. Like the gradebook's `check_score` in Week 12, it returns the clean value or raises `ValueError` with a message a person can act on.

## Step 2: showing money

The other direction, for printing. One function, used everywhere money is shown, so every amount in the program looks the same.

Notice the second check: a round-trip property from Week 24, Day 3. Whatever `format_pence` prints, `parse_amount` must be able to read back. Two functions written to agree with each other are easy to test together this way.

## Step 3: dates

Dates follow the advice from Week 24, Day 3: the function never asks the computer for today's date. It's **given** `today`, so its tests can pretend any day they like, and the real program passes `date.today()`.

## Step 4: categories

People type categories inconsistently: `Food`, `food `, `FOOD`. If they're stored as typed, a summary shows three categories where there's one. So, as in Week 23's Forge, clean the data **once, at the edge**, and everything after can trust it.

## Step 5: adding an expense

Now the first piece of `core.py`. Adding checks **every** field and reports **all** the problems at once, because a person who typed three things wrong would rather hear about all three than fix them one at a time. Only when everything is valid does anything change: an entry is either added complete or not at all.

## Step 6: removing an expense

## Checkpoint

Here's everything so far working together, as the interface will use it later this week: add some expenses, try a bad one, remove one, and add another to see that its id is new.

The last line shows ids 1, 2 and 4: number 3 was removed and is never handed out again.

(This checkpoint's `add_expense` loops over the three checks instead of writing three `try` blocks. Both work; tomorrow, with tests in place, you could choose either and know nothing broke.)

## Quiz

## Before you go

Make a folder called `spend` on your own computer, start a Git repository in it (Week 21), and put today's functions in `money.py` (amounts and categories), `dates.py` and `core.py` (`new_data`, `add_expense`, `remove_expense`), with the imports they need. Commit. Tomorrow: the rest of the core logic, filtering, totals, summaries and budgets, with a proper pytest suite.